# Demo: FP-Growth* — Khai thác Tập Phổ Biến




## 1. Nạp module

In [1]:
import Pkg
Pkg.activate(joinpath(@__DIR__, ".."))   # kích hoạt môi trường Project.toml
include(joinpath(@__DIR__, "..", "src", "FPGrowthStar.jl"))
using .FPGrowthStar
using Random
Random.seed!(42)   # cố định seed để kết quả tái lập được

  Activating project at `c:\Users\ADMIN\Desktop\itemset`


TaskLocalRNG()

## 2. Đọc dữ liệu & khai phá tập phổ biến



In [2]:
transactions = read_spmf(joinpath(@__DIR__, "..", "data", "toy", "example1.txt"))
println("Số giao dịch: ", length(transactions))
transactions

Số giao dịch: 7


7-element Vector{Vector{Int64}}:
 [1, 2, 3]
 [1, 2, 4]
 [1, 3, 4]
 [2, 3, 4]
 [1, 2, 3, 4]
 [1, 3]
 [2, 4]

In [3]:
minsup = 3
results = fpgrowth_star(transactions, minsup)
println("Số tập phổ biến (minsup=$minsup): ", length(results))
write_results(stdout, results)

Số tập phổ biến (minsup=3): 10
1 #SUP: 5
2 #SUP: 5
3 #SUP: 5
4 #SUP: 5
1 2 #SUP: 3
1 3 #SUP: 4
2 3 #SUP: 3
1 4 #SUP: 3
2 4 #SUP: 4
3 4 #SUP: 3


In [4]:
transactions2 = read_spmf(joinpath(@__DIR__, "..", "data", "toy", "example3.txt"))
println("Số giao dịch: ", length(transactions2))
transactions2
minsup = 3
results = fpgrowth_star(transactions, minsup)
println("Số tập phổ biến (minsup=$minsup): ", length(results))
write_results(stdout, results)

Số giao dịch: 5
Số tập phổ biến (minsup=3): 10
1 #SUP: 5
2 #SUP: 5
3 #SUP: 5
4 #SUP: 5
1 2 #SUP: 3
1 3 #SUP: 4
2 3 #SUP: 3
1 4 #SUP: 3
2 4 #SUP: 4
3 4 #SUP: 3


## 3. Trực quan hóa FP-tree

Cấu trúc dữ liệu cốt lõi của thuật toán. `build_fptree` dựng cây nén từ toàn bộ CSDL.

In [5]:
tree = build_fptree(transactions, minsup)
println("Thứ tự item (giảm dần theo support): ", tree.ordered_items)
println()
print_tree(tree.root)

Thứ tự item (giảm dần theo support): [1, 2, 3, 4]

root
  item=1 count=5
    item=2 count=3
      item=3 count=2
        item=4 count=1
      item=4 count=1
    item=3 count=2
      item=4 count=1
  item=2 count=2
    item=3 count=1
      item=4 count=1
    item=4 count=1


## 4. Trường hợp đặc biệt: FP-tree nhánh đơn (single-path)

Ví dụ 2 (`example2.txt`) là tình huống đặc biệt **single-path**: FP-tree chỉ có một nhánh. Khi đó FP-Growth\* không đệ quy nữa mà **liệt kê trực tiếp** mọi tập con khác rỗng của nhánh — đúng $2^n-1$ tập phổ biến (với $n$ là số item trên nhánh).

In [6]:
t2 = read_spmf(joinpath(@__DIR__, "..", "data", "toy", "example2.txt"))
r_star = fpgrowth_star(t2, 1)
n_items = length(unique(vcat(t2...)))
println("Số item phân biệt: ", n_items)
println("FP-Growth* tìm được: ", length(r_star), " tập phổ biến")
println("Kỳ vọng 2^n - 1   : ", 2^n_items - 1)
println("Khớp? ", length(r_star) == 2^n_items - 1)
write_results(stdout, r_star)

Số item phân biệt: 5
FP-Growth* tìm được: 31 tập phổ biến
Kỳ vọng 2^n - 1   : 31
Khớp? true
1 #SUP: 5
2 #SUP: 4
3 #SUP: 3
4 #SUP: 2
5 #SUP: 1
1 2 #SUP: 4
1 3 #SUP: 3
1 4 #SUP: 2
1 5 #SUP: 1
2 3 #SUP: 3
2 4 #SUP: 2
2 5 #SUP: 1
3 4 #SUP: 2
3 5 #SUP: 1
4 5 #SUP: 1
1 2 3 #SUP: 3
1 2 4 #SUP: 2
1 2 5 #SUP: 1
1 3 4 #SUP: 2
1 3 5 #SUP: 1
1 4 5 #SUP: 1
2 3 4 #SUP: 2
2 3 5 #SUP: 1
2 4 5 #SUP: 1
3 4 5 #SUP: 1
1 2 3 4 #SUP: 2
1 2 3 5 #SUP: 1
1 2 4 5 #SUP: 1
1 3 4 5 #SUP: 1
2 3 4 5 #SUP: 1
1 2 3 4 5 #SUP: 1


## 5. Chạy thử trên 2 tập benchmark chess.txt và mushroom.txt

Chạy thử với minsup = 10% để xem kết quả đối với 2 tập benchmark
.

In [7]:
transactions_chess = read_spmf(joinpath(@__DIR__, "..", "data", "benchmark", "chess.txt"))
println("Số giao dịch: ", length(transactions_chess))
transactions_chess
minsup = 0.9
results = fpgrowth_star(transactions_chess, minsup)
println("Số tập phổ biến (minsup=$minsup): ", length(results))
write_results(stdout, results)

Số giao dịch: 3196
Số tập phổ biến (minsup=0.9): 622
5 #SUP: 2971
7 #SUP: 3076
29 #SUP: 3181
34 #SUP: 3040
36 #SUP: 3099
40 #SUP: 3170
48 #SUP: 3013
52 #SUP: 3185
56 #SUP: 3021
58 #SUP: 3195
60 #SUP: 3149
62 #SUP: 3060
66 #SUP: 3021
5 29 #SUP: 2964
5 34 #SUP: 2889
5 40 #SUP: 2950
5 52 #SUP: 2960
5 58 #SUP: 2970
5 60 #SUP: 2924
7 29 #SUP: 3069
7 36 #SUP: 2979
7 40 #SUP: 3050
7 52 #SUP: 3065
7 58 #SUP: 3075
7 60 #SUP: 3031
29 52 #SUP: 3170
29 58 #SUP: 3180
7 34 #SUP: 2930
29 34 #SUP: 3036
34 36 #SUP: 2943
34 40 #SUP: 3017
34 52 #SUP: 3031
34 58 #SUP: 3039
34 60 #SUP: 2995
34 62 #SUP: 2913
29 36 #SUP: 3084
36 40 #SUP: 3073
36 52 #SUP: 3088
36 58 #SUP: 3098
36 60 #SUP: 3052
29 40 #SUP: 3155
40 52 #SUP: 3159
40 58 #SUP: 3169
7 48 #SUP: 2893
29 48 #SUP: 2998
36 48 #SUP: 2987
40 48 #SUP: 2987
48 52 #SUP: 3002
48 58 #SUP: 3012
48 60 #SUP: 2986
48 62 #SUP: 2877
52 58 #SUP: 3184
7 56 #SUP: 2919
29 56 #SUP: 3006
34 56 #SUP: 2884
36 56 #SUP: 2924
40 56 #SUP: 2996
52 56 #SUP: 3016
56 58 #SUP: 3020


In [8]:
transactions_mushroom = read_spmf(joinpath(@__DIR__, "..", "data", "benchmark", "mushroom.txt"))
println("Số giao dịch: ", length(transactions_mushroom))
transactions_mushroom
minsup = 0.60
results = fpgrowth_star(transactions_mushroom, minsup)
println("Số tập phổ biến (minsup=$minsup): ", length(results))
write_results(stdout, results)

Số giao dịch: 8416
Số tập phổ biến (minsup=0.6): 51
36 #SUP: 8200
38 #SUP: 6824
41 #SUP: 5880
67 #SUP: 5316
71 #SUP: 5076
90 #SUP: 8416
94 #SUP: 8216
97 #SUP: 7768
36 90 #SUP: 8200
36 94 #SUP: 8192
36 38 #SUP: 6608
38 90 #SUP: 6824
38 94 #SUP: 6632
38 97 #SUP: 6464
36 41 #SUP: 5664
41 90 #SUP: 5880
41 94 #SUP: 5688
41 97 #SUP: 5232
36 67 #SUP: 5124
67 90 #SUP: 5316
67 94 #SUP: 5124
71 90 #SUP: 5076
90 94 #SUP: 8216
36 97 #SUP: 7576
90 97 #SUP: 7768
94 97 #SUP: 7568
36 90 94 #SUP: 8192
36 38 90 #SUP: 6608
36 38 94 #SUP: 6608
38 90 94 #SUP: 6632
36 38 97 #SUP: 6272
38 90 97 #SUP: 6464
38 94 97 #SUP: 6272
36 41 90 #SUP: 5664
36 41 94 #SUP: 5664
41 90 94 #SUP: 5688
41 90 97 #SUP: 5232
36 67 94 #SUP: 5124
36 67 90 #SUP: 5124
67 90 94 #SUP: 5124
36 94 97 #SUP: 7568
36 90 97 #SUP: 7576
90 94 97 #SUP: 7568
36 38 90 94 #SUP: 6608
36 38 94 97 #SUP: 6272
36 38 90 97 #SUP: 6272
38 90 94 97 #SUP: 6272
36 41 90 94 #SUP: 5664
36 67 90 94 #SUP: 5124
36 90 94 97 #SUP: 7568
36 38 90 94 97 #SUP: 6272


## 6. Ứng dụng: sinh luật kết hợp (Market Basket Analysis)

Từ tập phổ biến, hàm `association_rules` của nhóm (`src/rules.jl`) sinh luật $X \Rightarrow Y$ với **support**, **confidence** và **lift**.

Dưới đây minh họa trên CSDL đồ chơi. Ứng dụng đầy đủ trên CSDL bán lẻ thực (Retail, 88K giao dịch → ~6000 luật) xem `test/test_rules.jl`, kết quả xuất ra `results/rules_retail.csv`.

In [9]:
rules = association_rules(results, length(transactions); minconf=0.6)
println("Số luật (minconf=0.6): ", length(rules))
println("\nTop luật kết hợp (theo lift):")
for r in first(rules, min(10, length(rules)))
    println("  ", r.antecedent, " => ", r.consequent,
            "  (sup=", r.support,
            ", conf=", round(r.confidence, digits=2),
            ", lift=", round(r.lift, digits=2), ")")
end

Số luật (minconf=0.6): 266

Top luật kết hợp (theo lift):
  [36, 38] => [94, 97]  (sup=6272, conf=0.95, lift=0.0)
  [36, 38] => [90, 94, 97]  (sup=6272, conf=0.95, lift=0.0)
  [36, 38, 90] => [94, 97]  (sup=6272, conf=0.95, lift=0.0)
  [94, 97] => [36, 38]  (sup=6272, conf=0.83, lift=0.0)
  [94, 97] => [36, 38, 90]  (sup=6272, conf=0.83, lift=0.0)
  [90, 94, 97] => [36, 38]  (sup=6272, conf=0.83, lift=0.0)
  [38, 94] => [36, 97]  (sup=6272, conf=0.95, lift=0.0)
  [38, 94] => [36, 90, 97]  (sup=6272, conf=0.95, lift=0.0)
  [38, 90, 94] => [36, 97]  (sup=6272, conf=0.95, lift=0.0)
  [36, 97] => [38, 94]  (sup=6272, conf=0.83, lift=0.0)
